#GOLD LAYER : BI READY

In [0]:
from pyspark.sql.types import StringType, IntegerType, DateType, BooleanType
import pyspark.sql.functions as F
from delta.tables import DeltaTable

dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
dbutils.widgets.text("storage_account_name", "storageaccount", "Storage Account Name")
dbutils.widgets.text("container_name", "ecomm-raw-data", "Container Name")

catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

# print(catalog_name, storage_account_name, container_name)

##FACT RETURNS

In [0]:
# Read incremental changes from Silver Returns

df_returns = (
    spark.readStream
    .format("delta")
    .option("readChangeFeed", "true")
    .table(f"{catalog_name}.silver.slv_order_returns")
)

In [0]:
# df_returns_static = spark.table(
#     f"{catalog_name}.silver.slv_order_returns"
# )

# df_returns_static.printSchema()

# display(df_returns_static.limit(10))

root
 |-- return_ts: timestamp (nullable = true)
 |-- order_dt: date (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- reason: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- ingest_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- processed_time: timestamp (nullable = true)



return_ts,order_dt,order_id,reason,_rescued_data,ingest_timestamp,source_file,processed_time
2025-05-01T02:11:57.000Z,2025-04-20,529233,OTHER,null,2026-06-15T07:02:21.059Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_returns/landing/returns_2025-05.csv,2026-06-15T10:02:06.667Z
2025-05-02T02:59:14.000Z,2025-04-17,525459,LATE DELIVERY,null,2026-06-15T07:02:21.059Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_returns/landing/returns_2025-05.csv,2026-06-15T10:02:06.667Z
2025-05-02T22:49:31.000Z,2025-04-20,528208,LATE DELIVERY,null,2026-06-15T07:02:21.059Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_returns/landing/returns_2025-05.csv,2026-06-15T10:02:06.667Z
2025-05-09T03:46:22.000Z,2025-04-25,534518,LATE DELIVERY,null,2026-06-15T07:02:21.059Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_returns/landing/returns_2025-05.csv,2026-06-15T10:02:06.667Z
2025-05-09T04:14:25.000Z,2025-04-22,530683,QUALITY ISSUE,null,2026-06-15T07:02:21.059Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_returns/landing/returns_2025-05.csv,2026-06-15T10:02:06.667Z
2025-05-18T13:40:04.000Z,2025-05-08,549444,LATE DELIVERY,null,2026-06-15T07:02:21.059Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_returns/landing/returns_2025-05.csv,2026-06-15T10:02:06.667Z
2025-05-22T23:21:36.000Z,2025-05-10,551636,DAMAGED,null,2026-06-15T07:02:21.059Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_returns/landing/returns_2025-05.csv,2026-06-15T10:02:06.667Z
2025-05-27T07:05:45.000Z,2025-05-19,562302,LATE DELIVERY,null,2026-06-15T07:02:21.059Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_returns/landing/returns_2025-05.csv,2026-06-15T10:02:06.667Z
2025-05-28T06:38:44.000Z,2025-05-18,560201,WRONG ITEM,null,2026-06-15T07:02:21.059Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_returns/landing/returns_2025-05.csv,2026-06-15T10:02:06.667Z
2025-01-05T05:11:05.000Z,2024-12-29,404276,WRONG ITEM,null,2026-06-15T07:02:21.059Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_returns/landing/returns_2025-01.csv,2026-06-15T10:02:06.667Z


In [0]:
# Keep only inserts and latest updates
df_returns_gold = df_returns.filter(
    "_change_type IN ('insert', 'update_postimage')"
)

df_returns_gold = (
    df_returns_gold

    # Date dimension key
    .withColumn(
        "date_id",
        F.date_format(
            F.col("order_dt"),
            "yyyyMMdd"
        ).cast("int")
    )

    # Days between order and return
    .withColumn(
        "return_days",
        F.datediff(
            F.to_date(F.col("return_ts")),
            F.col("order_dt")
        )
    )

    # Within return policy (15 days)
    .withColumn(
        "within_policy",
        F.when(
            F.col("return_days") <= 15,
            1
        ).otherwise(0)
    )

    # Late return flag
    .withColumn(
        "is_late_return",
        F.when(
            F.col("return_days") > 15,
            1
        ).otherwise(0)
    )
)

In [0]:
returns_gold_df = df_returns_gold.select(
    F.col("date_id"),
    F.col("order_id").alias("transaction_id"),
    F.col("order_dt").alias("order_date"),
    F.col("return_ts").alias("return_timestamp"),
    F.col("reason").alias("return_reason"),
    F.col("return_days"),
    F.col("within_policy"),
    F.col("is_late_return")
)

In [0]:
gold_checkpoint_path = (
    f"abfss://{container_name}@{storage_account_name}"
    f".dfs.core.windows.net/checkpoint/gold/fact_order_returns/"
)

def upsert_to_gold(microBatchDF, batchId):

    table_name = f"{catalog_name}.gold.gld_fact_order_returns"

    # First run
    if not spark.catalog.tableExists(table_name):

        print("Creating Gold table...")

        (
            microBatchDF.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(table_name)
        )

        spark.sql(
            f"""
            ALTER TABLE {table_name}
            SET TBLPROPERTIES
            (delta.enableChangeDataFeed = true)
            """
        )

    # Incremental loads
    else:

        deltaTable = DeltaTable.forName(
            spark,
            table_name
        )

        (
            deltaTable.alias("gold_table")
            .merge(
                microBatchDF.alias("batch_table"),
                """
                gold_table.transaction_id =
                batch_table.transaction_id

                AND

                gold_table.return_timestamp =
                batch_table.return_timestamp
                """
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

In [0]:
(
    returns_gold_df.writeStream
    .foreachBatch(upsert_to_gold)
    .option(
        "checkpointLocation",
        gold_checkpoint_path
    )
    .outputMode("update")
    .trigger(availableNow=True)
    .start()
    .awaitTermination()
)

##FACT SHIPMENT

In [0]:
# Read incremental changes from Silver Shipments

df_shipments = (
    spark.readStream
    .format("delta")
    .option("readChangeFeed", "true")
    .table(f"{catalog_name}.silver.slv_order_shipments")
)

In [0]:
# df_shipments_static = spark.table(
#     f"{catalog_name}.silver.slv_order_shipments"
# )

# df_shipments_static.printSchema()

# display(df_shipments_static.limit(10))

root
 |-- order_dt: date (nullable = true)
 |-- shipment_id: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- carrier: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- ingest_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- processed_time: timestamp (nullable = true)



order_dt,shipment_id,order_id,carrier,_rescued_data,ingest_timestamp,source_file,processed_time
2025-08-01,S643611,643611,BLUEDART,null,2026-06-15T07:30:13.815Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv,2026-06-15T10:06:20.587Z
2025-08-01,S643612,643612,AUSPOST,null,2026-06-15T07:30:13.815Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv,2026-06-15T10:06:20.587Z
2025-08-01,S643613,643613,DHL,null,2026-06-15T07:30:13.815Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv,2026-06-15T10:06:20.587Z
2025-08-01,S643614,643614,XPRESSBEES,null,2026-06-15T07:30:13.815Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv,2026-06-15T10:06:20.587Z
2025-08-01,S643615,643615,DELHIVERY,null,2026-06-15T07:30:13.815Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv,2026-06-15T10:06:20.587Z
2025-08-01,S643616,643616,BLUEDART,null,2026-06-15T07:30:13.815Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv,2026-06-15T10:06:20.587Z
2025-08-01,S643617,643617,DHL,null,2026-06-15T07:30:13.815Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv,2026-06-15T10:06:20.587Z
2025-08-01,S643618,643618,DHL,null,2026-06-15T07:30:13.815Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv,2026-06-15T10:06:20.587Z
2025-08-01,S643619,643619,DHL,null,2026-06-15T07:30:13.815Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv,2026-06-15T10:06:20.587Z
2025-08-01,S643620,643620,DHL,null,2026-06-15T07:30:13.815Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv,2026-06-15T10:06:20.587Z


In [0]:
# Keep only inserts and latest updates
df_shipments_gold = df_shipments.filter(
    "_change_type IN ('insert', 'update_postimage')"
)

df_shipments_gold = (
    df_shipments_gold

    # Date dimension key
    .withColumn(
        "date_id",
        F.date_format(
            F.col("order_dt"),
            "yyyyMMdd"
        ).cast("int")
    )

    # Carrier grouping
    .withColumn(
        "carrier_group",
        F.when(
            F.col("carrier").isin(
                "ECOMEXPRESS",
                "DELHIVERY",
                "XPRESSBEES",
                "BLUEDART"
            ),
            "Domestic"
        ).otherwise("International")
    )

    # Weekend shipment flag
    .withColumn(
        "is_weekend_shipment",
        F.when(
            F.dayofweek(F.col("order_dt")).isin(1, 7),
            True
        ).otherwise(False)
    )
)

In [0]:
shipments_gold_df = df_shipments_gold.select(
    F.col("date_id"),
    F.col("shipment_id"),
    F.col("order_id").alias("transaction_id"),
    F.col("order_dt").alias("shipment_date"),
    F.col("carrier"),
    F.col("carrier_group"),
    F.col("is_weekend_shipment")
)

In [0]:
gold_checkpoint_path = (
    f"abfss://{container_name}@{storage_account_name}"
    f".dfs.core.windows.net/checkpoint/gold/fact_order_shipments/"
)

def upsert_to_gold(microBatchDF, batchId):

    table_name = f"{catalog_name}.gold.gld_fact_order_shipments"

    # First run
    if not spark.catalog.tableExists(table_name):

        print("Creating Gold table...")

        (
            microBatchDF.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(table_name)
        )

        spark.sql(
            f"""
            ALTER TABLE {table_name}
            SET TBLPROPERTIES
            (delta.enableChangeDataFeed = true)
            """
        )

    # Incremental loads
    else:

        deltaTable = DeltaTable.forName(
            spark,
            table_name
        )

        (
            deltaTable.alias("gold_table")
            .merge(
                microBatchDF.alias("batch_table"),
                """
                gold_table.shipment_id =
                batch_table.shipment_id
                """
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

In [0]:
(
    shipments_gold_df.writeStream
    .foreachBatch(upsert_to_gold)
    .option(
        "checkpointLocation",
        gold_checkpoint_path
    )
    .outputMode("update")
    .trigger(availableNow=True)
    .start()
    .awaitTermination()
)